Multi-fidelity Modeling and Experimental Design (Active Learning)

In [1]:
# General imports
import numpy as np
np.random.seed(20)
import pandas as pd
from matplotlib import colors as mcolors
colors = dict(mcolors.BASE_COLORS, **mcolors.CSS4_COLORS)
import sys
import os
import drawing_utils_v8 as draw_mfsm
sys.path.append('../multi-fidelity-gaussian-process')
import multi_fidelity_surrogate_model_v8 as mfsm

from emukit.multi_fidelity.convert_lists_to_array import convert_x_list_to_array, convert_xy_lists_to_arrays
import random
import math
from sklearn.metrics import mean_squared_error

In [2]:
version = 'vmf1.6.1'
file_in=f'Ge77_rates_CNP_{version}.csv'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')

# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=['Radius[cm]','Thickness[cm]','NPanels', 'Theta[deg]', 'Length[cm]']
x_labels_out = ['Radius [cm]','Thickness [cm]','NPanels', 'Angle [deg]', 'Length [cm]']
y_label_cnp = 'Ge-77_CNP'
y_err_label_cnp = 'Ge-77_CNP_err'
y_label_sim = 'rGe77[nuc/(kg*yr)]'

# Set parameter boundaries
xmin=[0,0,0,0,0]
xmax=[265,20,360,90,150]

# Set parameter boundaries for aquisition function
xlow=[90,2,4,0,1]
xhigh=[250,15,360,90,150]

# Assign costs
low_fidelity_cost = 1.
high_fidelity_cost = 2000.

# Set a fixed point in space for drawings
x_fixed = [160, 2, 40, 45, 20]
# number of sigma for error band drawing on prediction
factor=1.

# Get LF noise from file
#with open(f'in/{file_in}') as f:
#    first_line = f.readline()
#LF_noise=np.round(float(first_line.split(' +')[0].split('= ')[1]),3)

# Get HF and LF data samples from file

data=pd.read_csv(f'in/{file_in}')
#data=data[[f'Mode', x_labels[0], x_labels[1], x_labels[2], x_labels[3], x_labels[4],y_label_cnp,y_err_label_cnp,y_label_sim]]

LF_noise=np.mean(data.loc[data['Mode']==0.][y_err_label_cnp].to_numpy())
HF_noise=np.mean(data.loc[data['Mode']==1.][y_err_label_cnp].to_numpy())

In [3]:

x_train_l, x_train_h, y_train_l, y_train_h = ([],[],[],[])
row_h=data.index[data['Mode'] == 1].tolist()
row_l=data.index[data['Mode'] == 0].tolist()

x_train_hf_sim = data.loc[data['Mode']==1.][x_labels].to_numpy().tolist()
y_train_hf_sim = data.loc[data['Mode']==1.][y_label_sim].to_numpy().tolist()

x_train_lf_sim = data.loc[data['Mode']==0.][x_labels].to_numpy().tolist()
y_train_lf_sim = data.loc[data['Mode']==0.][ y_label_sim].to_numpy().tolist()




In [4]:
%%capture
leg_label = []
ncol=1
nrow=int(np.ceil(len(x_labels)/ncol))


In [5]:


def model_validation(mf_model, x_train_hf_sim, y_train_hf_sim, x_labels, y_label, version):
        #data=pd.read_csv(file_in)
        #data=data[[f'Mode', x_labels[0], x_labels[1], x_labels[2], x_labels[3], x_labels[4],y_label]]

        #x_train_hf_sim = data.loc[data['Mode']==1.][x_labels].to_numpy().tolist()
        #y_train_hf_sim = data.loc[data['Mode']==1.][y_label].to_numpy().tolist()

        x_train_hf_sim, y_train_hf_sim = (np.atleast_2d(x_train_hf_sim), np.atleast_2d(y_train_hf_sim).T)

        counter_1sigma = 0
        counter_2sigma = 0
        counter_3sigma = 0

        mfsm_model_mean = np.empty(shape=[0, 0])
        mfsm_model_std = np.empty(shape=[0, 0])
        hf_data=[]
        x=[]
        MSE=0
        NMSE=0
        MAE=0
        MSSE=0
        for i in range(len(x_train_hf_sim)):

                SPLIT = 1
                x_plot = (np.atleast_2d(x_train_hf_sim[i]))
                X_plot = convert_x_list_to_array([x_plot , x_plot])
                hhf_mean_mf_model, hhf_var_mf_model = mf_model.predict(X_plot[1*SPLIT:])
                hhf_std_mf_model = np.sqrt(hhf_var_mf_model)

                hf_data.append(y_train_hf_sim[i][0])
                x.append(i)
                mfsm_model_mean=np.append(mfsm_model_mean,hhf_mean_mf_model[0,0])
                mfsm_model_std=np.append(mfsm_model_std,hhf_std_mf_model[0,0])

                if (y_train_hf_sim[i][0] < hhf_mean_mf_model[0][0]+hhf_std_mf_model[0][0]) and (y_train_hf_sim[i][0] > hhf_mean_mf_model[0][0]-hhf_std_mf_model[0][0]):
                        counter_1sigma += 1
                if (y_train_hf_sim[i][0] < hhf_mean_mf_model[0][0]+2*hhf_std_mf_model[0][0]) and (y_train_hf_sim[i][0] > hhf_mean_mf_model[0][0]-2*hhf_std_mf_model[0][0]):
                        counter_2sigma += 1
                if (y_train_hf_sim[i][0] < hhf_mean_mf_model[0][0]+3*hhf_std_mf_model[0][0]) and (y_train_hf_sim[i][0] > hhf_mean_mf_model[0][0]-3*hhf_std_mf_model[0][0]):
                        counter_3sigma += 1
                
                MAE +=np.abs(y_train_hf_sim[i][0]-hhf_mean_mf_model[0][0])
                MSE +=pow(y_train_hf_sim[i][0]-hhf_mean_mf_model[0][0],2)
                NMSE +=np.abs((y_train_hf_sim[i][0]-hhf_mean_mf_model[0][0])/hhf_std_mf_model[0][0])
                MSSE +=pow((y_train_hf_sim[i][0]-hhf_mean_mf_model[0][0])/hhf_std_mf_model[0][0],2)


        #if (counter_2sigma/len(hf_data)*100.==100. and counter_3sigma/len(hf_data)*100.==100. and counter_1sigma/len(hf_data)*100.<68.):
        #        counter_1sigma=counter_2sigma=counter_3sigma=0.
        percentage_1sigma=counter_1sigma/len(hf_data)*100.
        percentage_2sigma=counter_2sigma/len(hf_data)*100.
        percentage_3sigma=counter_3sigma/len(hf_data)*100.
        print("1 sigma: ", percentage_1sigma," %" )
        print("2 sigma: ", percentage_2sigma," %" )
        print("3 sigma: ", percentage_3sigma," %" )

        
        #fig = plt.subplots(figsize=(12, 2.5))
        ##plt.bar(x=np.arange(len(mfsm_model_mean)), height=mfsm_model_mean, color="lightgray", label='RESuM')
        #plt.fill_between(x=np.arange(len(mfsm_model_mean)), y1=mfsm_model_mean-3*mfsm_model_std, y2=mfsm_model_mean+3*mfsm_model_std, color="coral",alpha=0.2, label=r'$\pm 3\sigma$')
        #plt.fill_between(x=np.arange(len(mfsm_model_mean)), y1=mfsm_model_mean-2*mfsm_model_std, y2=mfsm_model_mean+2*mfsm_model_std, color="yellow",alpha=0.2, label=r'$\pm 2\sigma$')
        #plt.fill_between(x=np.arange(len(mfsm_model_mean)), y1=mfsm_model_mean-mfsm_model_std, y2=mfsm_model_mean+mfsm_model_std, color="green",alpha=0.2, label=r'RESuM $\pm 1\sigma$')
        #plt.xlabel('HF Simulation Trial Number')
        #plt.ylim(0.,0.55)
        #plt.ylabel(r'$y_{raw}$')
        #plt.plot(x[:],hf_data[:],'.',color="black", label="HF Validation Data")
        #handles, labels = plt.gca().get_legend_handles_labels()
        #order = [3,2,1,0]
        #plt.legend([handles[idx] for idx in order],[labels[idx] for idx in order],loc=9, bbox_to_anchor=(0.665,1.),ncol=5)
        #plt.savefig(f'out/{version}/model-validation_{version}.pdf')
        MAE=MAE/len(x_train_hf_sim)
        mse = mean_squared_error(hf_data,mfsm_model_mean, squared=True)
        NMSE=NMSE/len(x_train_hf_sim)
        MSSE=MSSE/len(x_train_hf_sim)
        return [percentage_1sigma,percentage_2sigma,percentage_3sigma,MAE,NMSE,mse,MSSE]

In [6]:

def run_all(start, end, mean, std, niter=100, n_restarts=100):
    for j in range(start,end):
        coverage=[]
        n_HF=j
        sample=0
        for i in range(niter):
            print('Sample #', sample)
            x_test_hf_sim=[]
            y_test_hf_sim=[]
            x_train_hf_sim = data.loc[data['Mode']==1.][x_labels].to_numpy().tolist()
            y_train_hf_sim = data.loc[data['Mode']==1.][y_label_sim].to_numpy().tolist()
            x_train_lf_sim = data.loc[data['Mode']==0.][x_labels].to_numpy().tolist()
            y_train_lf_sim = data.loc[data['Mode']==0.][ y_label_sim].to_numpy().tolist()
            
            x_train_lf_sim_tmp = x_train_lf_sim.copy()
            y_train_lf_sim_tmp = y_train_lf_sim.copy()
            x_train_hf_sim_tmp = x_train_hf_sim.copy()
            y_train_hf_sim_tmp = y_train_hf_sim.copy()
            # Generate a list of integers from 5 to 15
            indices = random.sample(range(4, 109), 106-n_HF+4)
            print(len(indices))
            for index in sorted(indices, reverse=True):
                x_train_lf_sim_tmp.pop(300+index)
                y_train_lf_sim_tmp.pop(300+index)
                x_train_hf_sim_tmp.pop(index)
                y_train_hf_sim_tmp.pop(index)
                x_test_hf_sim.append(x_train_hf_sim[index])
                y_test_hf_sim.append(y_train_hf_sim[index])
            x_test_hf_sim=x_test_hf_sim[:100]
            y_test_hf_sim=y_test_hf_sim[:100]
            print(len(y_train_lf_sim_tmp),len(y_train_hf_sim_tmp),len(y_train_hf_sim_tmp),len(y_test_hf_sim))

            x_train_lf_sim_tmp, x_train_hf_sim_tmp, y_train_lf_sim_tmp, y_train_hf_sim_tmp = (np.atleast_2d(x_train_lf_sim_tmp), np.atleast_2d(x_train_hf_sim_tmp),np.atleast_2d(y_train_lf_sim_tmp).T, np.atleast_2d(y_train_hf_sim_tmp).T)
            X_train, Y_train = convert_xy_lists_to_arrays([x_train_lf_sim_tmp,x_train_hf_sim_tmp], [y_train_lf_sim_tmp,y_train_hf_sim_tmp])

            mf_model = mfsm.linear_multi_fidelity_model(X_train, Y_train,[LF_noise,0.], 2, n_restarts)
            #draw_mfsm.draw_model(mf_model, xmin, xmax, x_labels_out, 2, version)#
            coverage.append(model_validation(mf_model,x_test_hf_sim,y_test_hf_sim,x_labels, y_label_sim, version))
            print(coverage)
            sample+=1
        # Convert the list to a NumPy array
        coverage = np.array(coverage)
        # Calculate the mean along axis 0 (i.e., column-wise mean)
        mean[n_HF]= np.mean(coverage, axis=0)
        std[n_HF]= np.std(coverage, axis=0)
    return [mean,std]

In [7]:
means =  np.zeros((11, 7))
std =  np.zeros((11, 7))
niters=50
nrestarts=10
np.set_printoptions(suppress=True, precision=3)

In [8]:
%%capture
means,std = run_all(5,6,means,std, niters, nrestarts)


In [9]:
%%capture
means,std = run_all(7,8,means, std,niters,nrestarts)

In [10]:
%%capture
means,std = run_all(10,11,means,std,niters,nrestarts)

In [11]:
# Open a file in write mode
with open(f"out/{version}/latex_table_{version}2.tex", "w") as file:
    # Write the LaTeX document header
    file.write("\\documentclass{article}\n")
    file.write("\\usepackage{booktabs}\n")
    file.write("\\usepackage{adjustbox}\n")
    file.write("\\begin{document}\n")
    file.write("\\begin{table}[ht]\n")
    file.write("\\centering\n")
    file.write("\\resizebox{\\textwidth}{!}{\n")
    file.write("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|}\n")
    file.write("\\hline\n")
    file.write("Model & no. HF & no. LF & $1\sigma$ & $1\sigma$ & $3\sigma$ & MAE & NMSE & MSE & MSSE & CPUh\\\\ \\hline\n")

    # Write each row of the table
    for i, row in enumerate(means):
        file.write(f"MF-BNN & {i} & {300+i} & {''.join([f'{x:.3f}+-{p:.3f} & ' for (x,p) in zip(means[i],std[i])])} \\\ \hline\n")

    # Write the LaTeX document footer
    file.write("\\end{tabular}\n")
    file.write("}\n")
    file.write("\\caption{MF-BNN}\n")
    file.write("\\label{tab:example}\n")
    file.write("\\end{table}\n")
    file.write("\\end{document}\n")

In [13]:
means

array([[ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 8.3  , 22.54 , 37.7  ,  0.04 ,  5.   ,  0.002, 40.038],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [30.54 , 51.5  , 64.7  ,  0.028,  3.429,  0.001, 33.364],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [44.22 , 69.88 , 82.14 ,  0.025,  1.93 ,  0.001, 10.804]])